# Second-stage NOG table generation from perturbed scenario configs

This notebook perturbs the base second-stage scenario, runs each configuration, and writes the generated optimization outputs into the NOC working directory.

The workflow is:
1. load the existing scenario config
2. mutate the initial state for each of the 12 perturbations
3. run the scenario for the nominal and perturbed cases
4. save the scenario outputs under the NOC folder
5. use those outputs to build the neighborhood-optimal gain table later

In [ ]:
from __future__ import annotations

from copy import deepcopy
from pathlib import Path

import numpy as np
import yaml

from space_traj_opt import CONFIG_DIR, OUT_DIR
from space_traj_opt.noc.noc_gain_generation import (
    STATE_NAMES,
    Trajectory,
    build_gain_table,
    save_gain_table,
)
from space_traj_opt.postprocessing.data_client import CSVClient
from space_traj_opt.sims.config import load_config
from space_traj_opt.sims.scenario import run_scenario

NOC_DIR = OUT_DIR / "noc"
STATE_ORDER = STATE_NAMES

## 1) Load the base second-stage scenario

This notebook starts from the existing scenario config in the repo and then perturbs the initial state vector. The generated files will be written directly into the NOC folder.

In [ ]:
scenario_path = CONFIG_DIR / "second_stage_ascent.yaml"
base_config = load_config(scenario_path)
print(base_config.name)
print(base_config.phases[0].state.guess)

deltas = {
    "x": 5000.0,
    "y": 5000.0,
    "z": 5000.0,
    "vx": 200.0,
    "vy": 200.0,
    "vz": 200.0,
}

## 2) Perturb the scenario config

This helper mutates the scenario initial state without changing the rest of the optimization setup. It mirrors the formal NOG perturbation set: + and - for each of the six states.

In [ ]:
def perturb_scenario_config(config, state_name: str, delta: float):
    """Return a deep-copied config with a single initial-state perturbation applied."""
    cfg = deepcopy(config)
    idx = STATE_ORDER.index(state_name)
    guess = np.asarray(cfg.phases[0].state.guess, dtype=float).copy()
    guess[idx] += delta
    cfg.phases[0].state.guess = guess.tolist()

    bounds = np.asarray(cfg.phases[0].state.bounds, dtype=float).copy()
    bounds[idx] += delta
    cfg.phases[0].state.bounds = bounds.tolist()

    cfg.name = f"{config.name}_{state_name}_{'plus' if delta >= 0 else 'minus'}"
    return cfg


def dump_config_yaml(cfg, filename: str):
    """Save the perturbed scenario config into the NOC folder."""
    out_path = CONFIG_DIR / "noc" / filename
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(yaml.safe_dump(cfg.model_dump(), sort_keys=False), encoding="utf-8")
    print(f"Saved config: {out_path}")
    return out_path

In [ ]:
nominal_cfg = deepcopy(base_config)
nominal_cfg.name = f"{base_config.name}_nominal"

configs = {"nominal": nominal_cfg}
for state_name in STATE_ORDER:
    for sign in (+1.0, -1.0):
        delta = sign * deltas[state_name]
        label = f"{state_name}_{'plus' if sign > 0 else 'minus'}"
        configs[label] = perturb_scenario_config(base_config, state_name, delta)

for label, cfg in configs.items():
    dump_config_yaml(cfg, f"{cfg.name}.yaml")

sorted(configs)

## 3) Run each scenario and dump outputs into the NOC folder

This executes the actual optimization for the nominal and each perturbed configuration. The generated CSV/JSON metadata files will be written under the NOC folder rather than the default output directory.

In [ ]:
if True:
        
    for label, cfg in configs.items():
        print(f"Running scenario: {cfg.name}")
        result = run_scenario(cfg, "noc")
        print(f"  status: {result.success}")
        print(f"  objective: {result.fun}")



## 4) Build the NOG table from the generated runs

Once the nominal and perturbed trajectories are generated and stored in the NOC folder, you can load those outputs and construct a gain table using the same finite-difference formulation in the NOG code.

In [ ]:

def load_trajectory_from_csv(csv_path: Path) -> Trajectory:
    """Load a simulation CSV into the NOC trajectory format."""
    client = CSVClient(f"noc/{csv_path.stem}")

    state = client.get_channels(["rx", "ry", "rz", "vx", "vy", "vz"])
    #acceleration = np.linalg.norm(client.get_channels(["ax", "ay", "az"]), axis=1)
    acceleration = -client.get_channels(["m"])
    control = client.get_channels(["pitch", "yaw"])

    return Trajectory(
        acceleration=acceleration,
        state=state,
        control=control,
    )


csv_files = sorted(NOC_DIR.glob("*.csv"))
trajectory_map = {path.stem: load_trajectory_from_csv(path) for path in csv_files}

nominal_name = next(name for name in trajectory_map if "nominal" in name)
nominal = trajectory_map[nominal_name]

perturbations = {}
for state_name in STATE_ORDER:
    for sign, label in ((+1.0, "+"), (-1.0, "-")):
        csv_name = f"second_stage_ascent_{state_name}_{'plus' if sign > 0 else 'minus'}"
        print(csv_name)
        perturbations[f"{label}{state_name}"] = trajectory_map[csv_name]





In [ ]:
acceleration_grid = np.linspace(
    nominal.acceleration.min(),
    nominal.acceleration.max(),
    50,
)

gain_table = build_gain_table(
    nominal=nominal,
    perturbations=perturbations,
    deltas=deltas,
    acceleration_grid=acceleration_grid,
)

save_gain_table(gain_table, NOC_DIR / "second_stage_ascent_gain_table.npz")
print(f"nominal: {nominal_name}")
print(f"gain shape: {gain_table.gain.shape}")
print(f"saved: {NOC_DIR / 'second_stage_ascent_gain_table.npz'}")
